# eos-ml — Quickstart (interactive)

This notebook walks through the `eos-ml` pipeline end-to-end on synthetic data and provides lightweight interactive widgets (via `ipywidgets`) to explore posterior samples, M–R curves, and point estimates.

**Prerequisites:** install dependencies from `requirements.txt` and `ipywidgets`. Run the notebook from the `eos-ml/notebooks/` directory so imports from the parent directory work (the notebook will add the parent folder to `sys.path`).

In [1]:
import os, sys
# make sure parent project folder is on sys.path (notebook placed in notebooks/ )
sys.path.insert(0, os.path.abspath('..'))
print('Project root added to sys.path:', sys.path[0])

Project root added to sys.path: c:\Users\sarva\OneDrive\Desktop\eos-ml


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import ipywidgets as widgets
plt.rcParams['figure.figsize'] = (6,4) 
# core pipeline imports (from parent folder)
from inference.dataset import synth_dataset
from inference.sampler import make_eos
from inference.supervised_baseline import train_inverse_regressor, predict_inverse
from inference.sbi_flows import train_flow, flow_sample_posterior
from physics.forward_model import ForwardModel
from verify.plots import plot_mr_curve
print('Imports OK')

Imports OK


In [3]:
# Generate a small synthetic dataset for interactive demo (fast)
X, TH = synth_dataset(N=60, seed=2)
print('Generated dataset shapes — X, TH:', X.shape, TH.shape)
display(Markdown('**First 5 observables (M [M_sun], R [km], Lambda):**'))
print(X[:5])

1
iteration 0  current length 0
sample theta
make eos
causality func
monotone func
Max mass ok func
4931799.634455641     1.0284710201857581e-54    50.0
EoS accepted : {'rho1': 89241820137397.47, 'rho2': 379094686048474.0, 'Gamma1': 2.2584934627725644, 'Gamma2': 1.4114066669107228, 'Gamma3': 2.7602613675107, 'P1': 1.729029811743109e+33}
iteration 1  current length 1
sample theta
make eos
causality func
monotone func
Rejected: monotonicity
iteration 2  current length 1
sample theta
make eos
causality func
monotone func
Rejected: monotonicity
iteration 3  current length 1
sample theta
make eos
causality func
monotone func
Rejected: monotonicity
iteration 4  current length 1
sample theta
make eos
causality func
monotone func
Rejected: monotonicity
iteration 5  current length 1
sample theta
make eos
causality func
monotone func
Rejected: monotonicity
iteration 6  current length 1
sample theta
make eos
causality func
monotone func
Max mass ok func
4894220.370306579     1.0685665568854432e-5

**First 5 observables (M [M_sun], R [km], Lambda):**

[[4.93179963e+06 5.00000000e+01 1.02847102e-54]
 [4.89422037e+06 5.00000000e+01 1.06856656e-54]
 [4.88310741e+06 5.00000000e+01 1.08078122e-54]
 [5.06072236e+06 5.00000000e+01 9.03975419e-55]
 [5.03152820e+06 5.00000000e+01 9.30506956e-55]]


In [4]:
# Train a fast inverse regressor (MLP) — fewer epochs for demo speed\n",
print('Training inverse regressor (demo: epochs=60) ...')
inv = train_inverse_regressor(X, TH, epochs=60, verbose=True)
display(Markdown('Trained inverse regressor (object `inv`)'))

Training inverse regressor (demo: epochs=60) ...
[inv] epoch 50/60 mse=inf


Trained inverse regressor (object `inv`)

In [5]:
# Train a conditional flow (SBI) — reduced epochs for demo\n",
print('Training conditional flow (demo: epochs=80) ...')
flow = train_flow(TH, X, epochs=80, verbose=True)
display(Markdown('Trained conditional flow (object `flow`)'))

Training conditional flow (demo: epochs=80) ...
[flow] epoch 50/80 nll=nan


Trained conditional flow (object `flow`)

In [6]:
# Interactive view function: shows point estimate, true EoS, and posterior samples' M-R curves + simple param histogram
def interactive_view(idx=0, n_post=100, n_mr_samples=30):
    theta_keys = ["rho1","rho2","Gamma1","Gamma2","Gamma3","P1"]
    x_obs = X[idx]
    display(Markdown(f'**Selected observation index:** {idx} — x_obs = {x_obs}'))
    # point estimate (inverse regressor)
    th_map = predict_inverse(inv, x_obs.reshape(1, -1))[0]
    th_est = {k: float(v) for k, v in zip(theta_keys, th_map)}
    eos_est = make_eos(th_est)
    fm_est = ForwardModel(eos_est)
    # ground-truth EoS used to generate the synthetic obs
    th_true = {k: float(v) for k, v in zip(theta_keys, TH[idx])}
    eos_true = make_eos(th_true)
    fm_true = ForwardModel(eos_true)

    # plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    rho_scan = np.geomspace(6e13, 3e15, 40)

    # M-R curves: true, estimate
    plot_mr_curve(fm_true, rho_scan, ax=axes[0], label='true')
    plot_mr_curve(fm_est, rho_scan, ax=axes[0], label='estimate')

    # posterior samples from flow (theta space), plot subset of MR curves
    S = flow_sample_posterior(flow, x_obs, n_samples=n_post)
    n_plot = min(n_mr_samples, S.shape[0])
    for s in S[:n_plot]:
        try:
            th_s = {k: float(v) for k, v in zip(theta_keys, s)}
            eos_s = make_eos(th_s)
            fm_s = ForwardModel(eos_s)
            Ms, Rs = [], []
            for rc in rho_scan:
                M, R = fm_s.tov.integrate_star(rc)
                Ms.append(M/1.98847e33)  # Msun
                Rs.append(R/1e5)         # km
            axes[0].plot(Rs, Ms, color='gray', alpha=0.25, lw=0.8)
        except Exception:
            continue

    axes[0].set_title('M–R: true vs estimate + posterior samples')
    axes[0].legend()

    # histogram of one posterior parameter (Gamma2)
    gamma2_vals = S[:, 3]
    axes[1].hist(gamma2_vals, bins=25, alpha=0.8)
    axes[1].axvline(th_true['Gamma2'], color='C1', label='true Gamma2')
    axes[1].axvline(th_est['Gamma2'], color='C2', label='map est')
    axes[1].set_xlabel('Gamma2')
    axes[1].legend()
    plt.show()

In [7]:
# Interactive widgets: index slider, posterior sample size, and how many MR curves to overlay,
widgets.interact(
    interactive_view,
    idx=widgets.IntSlider(min=0, max=len(X)-1, step=1, value=0),
    n_post=widgets.IntSlider(min=20, max=500, step=10, value=100),
    n_mr_samples=widgets.IntSlider(min=0, max=100, step=1, value=30)
)

interactive(children=(IntSlider(value=0, description='idx', max=59), IntSlider(value=100, description='n_post'…

<function __main__.interactive_view(idx=0, n_post=100, n_mr_samples=30)>

## Next steps and notes
- This notebook is a compact interactive demo: for production-quality runs increase dataset size and training epochs (and use GPU).
- To use **real NICER / GW** data: place CSVs in `../data/` and add a loader cell (I can add that for you).
- If ipywidgets don't appear, enable the extension (`jupyter nbextension enable --py widgetsnbextension`) or use JupyterLab extension for widgets.
Tell me if you want: (A) a cell that loads NICER/GW CSVs and builds the likelihood, (B) a version that uses a crust model + validated Love-number integrator, or (C) export cells to a downloadable `.ipynb` file placed in the repo."